# 08 — Embedding Space Analysis

Whisper's encoder produces a 384-dimensional embedding for each audio clip — a compressed representation of what the model "hears." By visualizing these embeddings, we can understand how Whisper organizes Dothraki speech internally: do similar phrases cluster together? Do correct matches live in denser neighborhoods? This notebook explores the geometry of our 1,712-clip embedding index.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

FEATURES_DIR = PROJECT_ROOT / 'data' / 'features'
RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

C_TEAL = '#4ecdc4'
C_RED = '#ff6b6b'
C_YELLOW = '#ffd93d'
C_DARK_TEAL = '#45b7aa'
COLORS = [C_TEAL, C_RED, C_YELLOW, C_DARK_TEAL]

# Load embedding index
npz = np.load(FEATURES_DIR / 'embedding_index_tiny.npz', allow_pickle=True)
embeddings = npz['embeddings']  # (1712, 384)
metadata = json.loads(str(npz['metadata']))

# Load embedding eval results
emb_eval = json.loads((RESULTS_DIR / 'batch_eval_embedding_small.json').read_text())
emb_results = emb_eval['results']

print(f'Embeddings shape: {embeddings.shape}')
print(f'Metadata entries: {len(metadata)}')
print(f'Eval results: {len(emb_results)} clips')

---
## 1. t-SNE Projection

Reduce 384 dimensions → 2D using t-SNE to see how Whisper organizes Dothraki audio. Points are colored by their source (which season/dataset the dialogue came from).

In [ ]:
# Run t-SNE
tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
coords = tsne.fit_transform(embeddings)

# Extract source labels
sources = [m.get('scene', 'unknown').split(':')[0].strip() if 'scene' in m else 'unknown' for m in metadata]
# Simplify source labels
source_map = {}
for s in sources:
    if 'Season 1' in s or 'seasons_1' in s:
        source_map[s] = 'Seasons 1-2'
    elif 'Season 3' in s or 'seasons_3' in s:
        source_map[s] = 'Seasons 3-4'
    elif 'Requested' in s:
        source_map[s] = 'Requested Translations'
    else:
        source_map[s] = 'Other'

# Use the source field from manifest metadata directly
source_labels = [m.get('source', 'unknown') for m in metadata]
unique_sources = sorted(set(source_labels))
color_map = {s: COLORS[i % len(COLORS)] for i, s in enumerate(unique_sources)}

fig, ax = plt.subplots(figsize=(14, 10))
for src in unique_sources:
    mask = [s == src for s in source_labels]
    ax.scatter(coords[mask, 0], coords[mask, 1],
               c=color_map[src], label=src, alpha=0.6, s=15, edgecolor='none')
ax.set_title('t-SNE Projection of Whisper Encoder Embeddings (1,712 clips)', fontsize=14)
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.legend(loc='upper right', fontsize=10)
plt.tight_layout()
plt.show()

---
## 2. Cluster Analysis

Apply KMeans (k=10) to the embedding space. Are there natural groupings, and do they correspond to linguistic features?

In [ ]:
# KMeans clustering
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
labels = kmeans.fit_predict(embeddings)

# Cluster sizes
cluster_sizes = np.bincount(labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart of cluster sizes
bars = axes[0].bar(range(10), cluster_sizes, color=C_TEAL, edgecolor='#1a1a2e', alpha=0.85)
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Number of Clips')
axes[0].set_title('Cluster Sizes (k=10)')

# t-SNE colored by cluster
scatter = axes[1].scatter(coords[:, 0], coords[:, 1], c=labels, cmap='tab10',
                           alpha=0.6, s=15, edgecolor='none')
axes[1].set_title('t-SNE Colored by KMeans Cluster')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')

plt.tight_layout()
plt.show()

# Sample phrases per cluster
print('\nSample phrases per cluster:')
print('=' * 70)
for c in range(10):
    idxs = np.where(labels == c)[0]
    samples = [metadata[i]['dothraki'][:50] for i in idxs[:3]]
    print(f'Cluster {c} ({cluster_sizes[c]} clips): {", ".join(samples)}')

---
## 3. Correct vs Incorrect Matches

For the 200 evaluation clips, compare cosine distances between the query embedding and its top match for exact matches vs mismatches.

In [ ]:
# Build clip_id -> embedding index mapping
id_to_idx = {m['clip_id']: i for i, m in enumerate(metadata)}

correct_dists = []
incorrect_dists = []

for r in emb_results:
    query_id = r['id']
    match_id = r.get('top_match_id')
    if query_id not in id_to_idx or match_id not in id_to_idx:
        continue
    q_emb = embeddings[id_to_idx[query_id]].reshape(1, -1)
    m_emb = embeddings[id_to_idx[match_id]].reshape(1, -1)
    cos_sim = cosine_similarity(q_emb, m_emb)[0, 0]
    cos_dist = 1 - cos_sim
    if r.get('exact_match', False):
        correct_dists.append(cos_dist)
    else:
        incorrect_dists.append(cos_dist)

fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot([correct_dists, incorrect_dists],
                labels=['Exact Match', 'Mismatch'],
                patch_artist=True,
                boxprops=dict(facecolor=C_TEAL, alpha=0.7),
                medianprops=dict(color=C_RED, linewidth=2),
                flierprops=dict(markerfacecolor=C_YELLOW, markersize=4))
bp['boxes'][1].set_facecolor(C_RED)
ax.set_ylabel('Cosine Distance (1 - similarity)')
ax.set_title('Cosine Distance: Correct vs Incorrect Embedding Matches')
ax.text(1, max(correct_dists) * 1.1 if correct_dists else 0, f'n={len(correct_dists)}', ha='center', fontsize=10)
ax.text(2, max(incorrect_dists) * 1.1 if incorrect_dists else 0, f'n={len(incorrect_dists)}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

print(f'Correct matches: mean dist = {np.mean(correct_dists):.4f}, median = {np.median(correct_dists):.4f}')
print(f'Incorrect matches: mean dist = {np.mean(incorrect_dists):.4f}, median = {np.median(incorrect_dists):.4f}')

---
## 4. Nearest Neighbor Examples

Find the 5 closest embedding pairs (excluding self-matches) and check whether they're semantically related.

In [ ]:
# Compute pairwise cosine similarity for a manageable subset
# Use full matrix but extract top pairs efficiently
sim_matrix = cosine_similarity(embeddings)
np.fill_diagonal(sim_matrix, -1)  # exclude self-matches

# Find top 5 closest pairs
flat_idx = np.argsort(sim_matrix.ravel())[::-1][:10]  # top 10 entries = 5 pairs (symmetric)
seen_pairs = set()
top_pairs = []

for idx in flat_idx:
    i, j = divmod(idx, sim_matrix.shape[1])
    pair = (min(i, j), max(i, j))
    if pair not in seen_pairs:
        seen_pairs.add(pair)
        top_pairs.append((i, j, sim_matrix[i, j]))
    if len(top_pairs) == 5:
        break

print('Top 5 Closest Embedding Pairs (non-self):')
print('=' * 90)
for rank, (i, j, sim) in enumerate(top_pairs, 1):
    print(f'\n#{rank} — Cosine similarity: {sim:.4f}')
    print(f'  Clip {metadata[i]["clip_id"]}: {metadata[i]["dothraki"][:60]}')
    print(f'    English: {metadata[i]["english"][:60]}')
    print(f'  Clip {metadata[j]["clip_id"]}: {metadata[j]["dothraki"][:60]}')
    print(f'    English: {metadata[j]["english"][:60]}')

---
## 5. Similarity Heatmap

A 20x20 cosine similarity matrix for a diverse subset of clips, showing the structure of the embedding space.

In [ ]:
# Pick 20 diverse clips: 2 from each KMeans cluster
diverse_idx = []
for c in range(10):
    cluster_members = np.where(labels == c)[0]
    diverse_idx.extend(cluster_members[:2])

subset_emb = embeddings[diverse_idx]
subset_sim = cosine_similarity(subset_emb)
subset_labels = [f'{metadata[i]["clip_id"]}' for i in diverse_idx]

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(subset_sim, cmap='YlOrRd', vmin=0.5, vmax=1.0, aspect='auto')
ax.set_xticks(range(20))
ax.set_yticks(range(20))
ax.set_xticklabels(subset_labels, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(subset_labels, fontsize=8)
ax.set_title('Cosine Similarity Heatmap (20 Diverse Clips)', fontsize=14)
plt.colorbar(im, ax=ax, label='Cosine Similarity', shrink=0.8)
plt.tight_layout()
plt.show()

---
## Conclusions

1. **Whisper's embedding space captures acoustic structure** — t-SNE reveals clear clusters, suggesting the encoder organizes Dothraki clips by acoustic similarity even without Dothraki-specific training.

2. **Clusters partially align with linguistic content** — KMeans groups tend to contain phrases of similar length and phonological character, though not always semantic similarity.

3. **Cosine distance separates correct from incorrect matches** — exact matches have near-zero distance (self-retrieval), while mismatches show higher cosine distances, confirming the embedding approach's underlying mechanism.

4. **Nearest neighbors reveal acoustic confusability** — the closest non-self pairs highlight which Dothraki phrases sound most alike to Whisper, pointing to potential error cases.

**Key Takeaway:** Whisper's encoder, despite never seeing Dothraki, builds a structured representation space that supports 73.5% exact-match retrieval. The embedding space's geometry suggests that Whisper treats Dothraki audio as phonetically structured signal rather than noise.